## Using Estimates from Variational, Laplace, or Optimization Methods to Initialize the NUTS-HMC Sampler

In this example, we show how to use parameter estimates returned by Stan's various posterior approximation or optimization algorithms as initial values for Stan's NUTS-HMC sampler. These include:

* [Pathfinder ](https://mc-stan.org/docs/cmdstan-guide/pathfinder-config.html) 
* [ADVI ](https://mc-stan.org/docs/cmdstan-guide/variational_config.html) 
* [Laplace](https://mc-stan.org/docs/cmdstan-guide/laplace_sample_config.html)
* [Optimization](https://mc-stan.org/docs/cmdstan-guide/optimize_config.html)

By default, the NUTS-HMC sampler randomly initializes all model parameters uniformly in the interval $(-2, 2)$.  If this interval is far from the typical set of the posterior, initializing sampling from these approximation algorithms can speed up and improve adaptation.

### Model and data

The Stan model and data are taken from the [posteriordb package](https://github.com/stan-dev/posteriordb).

We use the [blr model](https://github.com/stan-dev/posteriordb/blob/master/posterior_database/models/stan/blr.stan),
a Bayesian standard linear regression model with noninformative priors,
and its corresponding simulated dataset [sblri.json](https://github.com/stan-dev/posteriordb/blob/master/posterior_database/data/data/sblri.json.zip),
which was simulated via script [sblr.R](https://github.com/stan-dev/posteriordb/blob/master/posterior_database/data/data-raw/sblr/sblr.R).
For convenience, this example assumes the posteriordb model and data are local, in files `blr.stan` and `sblri.json`.

In [1]:
import os
from cmdstanpy import CmdStanModel

stan_file = 'blr.stan' # basic linear regression
data_file = 'sblri.json' # simulated data

model = CmdStanModel(stan_file=stan_file)

print(model.code())

data {
  int<lower=0> N;
  int<lower=0> D;
  matrix[N, D] X;
  vector[N] y;
}
parameters {
  vector[D] beta;
  real<lower=0> sigma;
}
model {
  // prior
  target += normal_lpdf(beta | 0, 10);
  target += normal_lpdf(sigma | 0, 10);
  // likelihood
  target += normal_lpdf(y | X * beta, sigma);
}




### Demonstration with Stan's `pathfinder` method

The approximation methods all follow the same general pattern of usage. First, we call the 
corresponding method on the `CmdStanModel` object. From the resulting fit, we call the `.create_inits()` 
method to construct a set of per-chain initializations for the model parameters. To make it explicit, 
we will walk through the process using the `pathfinder` method (which wraps the 
CmdStan [pathfinder ](https://mc-stan.org/docs/cmdstan-guide/pathfinder-config.html) method).

Pathfinder locates normal approximations to the target
density along a quasi-Newton optimization path, with local covariance
estimated using the negative inverse Hessian estimates produced by the
LBFGS optimizer. Pathfinder returns draws from the Gaussian approximation
with the lowest estimated Kullback-Leibler (KL) divergence to the true
posterior.
By default, CmdStanPy runs multi-path Pathfinder which returns an importance-resampled 
set of draws over the outputs of 4 independent single-path Pathfinders.
This better matches non-normal target densities and also mitigates
the problem of L-BFGS getting stuck at local optima or in saddle points on plateaus.

We obtain Pathfinder estimates by calling the `.pathfinder()` method which returns a `CmdStanPathfinder` object:

In [2]:
pathfinder_fit = model.pathfinder(data=data_file, seed=123)

21:47:10 - cmdstanpy - INFO - Chain [1] start processing
21:47:10 - cmdstanpy - INFO - Chain [1] done processing


Posteriordb provides reference posteriors for all models. For the blr model, conditioned on the dataset `sblri.json`, the reference posteriors can be found in the [sblri-blr.json](https://github.com/stan-dev/posteriordb/blob/master/posterior_database/reference_posteriors/summary_statistics/mean/mean/sblri-blr.json) file.

The reference posteriors for all elements of `beta` and `sigma` are all very close to $1.0$.

The experiments reported in Figure 3 of the paper [Pathfinder: Parallel quasi-Newton variational inference](https://arxiv.org/abs/2108.03782) by Zhang et al. show that Pathfinder provides a better estimate of the posterior, as measured by the 1-Wasserstein distance to the reference posterior, than 75 iterations of the warmup Phase I algorithm used by the NUTS-HMC sampler.
Furthermore, Pathfinder is more computationally efficient, requiring fewer evaluations of the log density and gradient functions. Therefore, using the Pathfinder estimates to initialize the parameter values for the NUTS-HMC sampler can allow the sampler to do a better job of adapting the stepsize and metric during warmup, resulting in better performance and estimation.

We construct the parameter inits for full MCMC sampling below. The `.create_inits()` default behavior is to create inits for four chains to correspond with the sampling defaults. You can requests more or less by modifying the `chains` keyword argument.

In [3]:
pathfinder_inits = pathfinder_fit.create_inits()
for chain_init in pathfinder_inits:
    print(chain_init)

{'beta': array([1.00019 , 0.999721, 0.999143, 1.00221 , 1.00193 ]), 'sigma': array(0.955428)}
{'beta': array([0.998311, 1.00282 , 1.00017 , 1.00119 , 1.00148 ]), 'sigma': array(0.829495)}
{'beta': array([1.0007  , 1.00177 , 0.999522, 1.00289 , 0.999926]), 'sigma': array(0.904491)}
{'beta': array([0.998958, 1.00013 , 1.00095 , 0.999549, 1.00184 ]), 'sigma': array(0.895088)}


We see that the Pathfinder inits are close the reference posteriors for the parameters. To use these inits, we pass the `pathfinder_inits` object to the `inits` kwarg:

In [4]:
mcmc_pathfinder_inits_fit = model.sample(
    data=data_file, inits=pathfinder_inits, iter_warmup=75, seed=12345
)

21:47:10 - cmdstanpy - INFO - CmdStan start processing


chain 1:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 2:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 3:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 4:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

21:47:10 - cmdstanpy - INFO - CmdStan done processing.


In [5]:
print(mcmc_pathfinder_inits_fit.diagnose())

Processing csv files: /tmp/tmplo9k1zwz/blr_twoiy_n/blr-20250507214710_1.csv, /tmp/tmplo9k1zwz/blr_twoiy_n/blr-20250507214710_2.csv, /tmp/tmplo9k1zwz/blr_twoiy_n/blr-20250507214710_3.csv, /tmp/tmplo9k1zwz/blr_twoiy_n/blr-20250507214710_4.csv

Checking sampler transitions treedepth.
Treedepth satisfactory for all transitions.

Checking sampler transitions for divergences.
No divergent transitions found.

Checking E-BFMI - sampler transitions HMC potential energy.
E-BFMI satisfactory.

Effective sample size satisfactory.

Split R-hat values satisfactory all parameters.

Processing complete, no problems detected.



Despite only running 75 warmup iterations, all posterior diagnostics from the sampler look good.

In [6]:
mcmc_pathfinder_inits_fit.summary()

,Mean,MCSE,StdDev,5%,50%,95%,N_Eff,N_Eff/s,R_hat
lp__,-156.923000,0.069791,1.822880,-160.397000,-156.590000,-154.59900,682.21200,3832.65000,1.009740
beta[1],0.999475,0.000014,0.000979,0.997822,0.999471,1.00111,4972.69000,27936.50000,0.999750
beta[2],1.000240,0.000018,0.001169,0.998325,1.000220,1.00216,4377.88000,24594.90000,0.999390
beta[3],1.000430,0.000014,0.000957,0.998818,1.000440,1.00198,4804.90000,26993.80000,0.999950
beta[4],1.001150,0.000016,0.001062,0.999448,1.001150,1.00291,4289.55000,24098.60000,0.999605
beta[5],1.001580,0.000015,0.001038,0.999896,1.001570,1.00328,4676.14000,26270.40000,1.000200
sigma,0.962170,0.004210,0.071330,0.854270,0.959130,1.08159,286.75663,1610.99231,1.010100


If we were to instead use the default random parameter initializations, we would need to run more warmup iterations to produce useful samples. For example, if we only run the same 75 warmup iterations with random inits, the result fails to estimate `sigma` correctly: 

In [7]:
mcmc_random_inits_fit = model.sample(data=data_file, iter_warmup=75, seed=12345)

21:47:11 - cmdstanpy - INFO - CmdStan start processing


chain 1:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 2:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 3:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 4:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

21:47:11 - cmdstanpy - INFO - CmdStan done processing.
21:47:11 - cmdstanpy - WARNING - Some chains may have failed to converge.
	Chain 1 had 177 divergent transitions (17.7%)
	Chain 3 had 173 divergent transitions (17.3%)
	Chain 4 had 247 divergent transitions (24.7%)
	Use the "diagnose()" method on the CmdStanMCMC object to see further information.


In [8]:
mcmc_random_inits_fit.summary()

,Mean,MCSE,StdDev,5%,50%,95%,N_Eff,N_Eff/s,R_hat
lp__,-191.928000,24.770200,35.245800,-232.300000,-165.541000,-155.46100,2.02467,35.52060,12.75920
beta[1],0.999859,0.000148,0.002086,0.996608,0.999924,1.00357,198.16000,3476.48000,1.03553
beta[2],0.999949,0.000266,0.002675,0.996096,0.999525,1.00457,101.40000,1778.95000,1.04521
beta[3],1.000480,0.000147,0.002187,0.996966,1.000410,1.00452,222.45200,3902.66000,1.00934
beta[4],1.001410,0.000177,0.002659,0.996444,1.001660,1.00611,225.98700,3964.68000,1.03112
beta[5],1.001680,0.000192,0.002525,0.997740,1.001290,1.00655,173.71700,3047.67000,1.03154
sigma,1.978920,0.714010,1.018880,0.917490,2.700570,3.17346,2.03627,35.72403,10.27506


In [9]:
print(mcmc_random_inits_fit.diagnose())

Processing csv files: /tmp/tmplo9k1zwz/blr8kgp791v/blr-20250507214711_1.csv, /tmp/tmplo9k1zwz/blr8kgp791v/blr-20250507214711_2.csv, /tmp/tmplo9k1zwz/blr8kgp791v/blr-20250507214711_3.csv, /tmp/tmplo9k1zwz/blr8kgp791v/blr-20250507214711_4.csv

Checking sampler transitions treedepth.
Treedepth satisfactory for all transitions.

Checking sampler transitions for divergences.
597 of 4000 (14.93%) transitions ended with a divergence.
These divergent transitions indicate that HMC is not fully able to explore the posterior distribution.
Try increasing adapt delta closer to 1.
If this doesn't remove all divergences, try to reparameterize the model.

Checking E-BFMI - sampler transitions HMC potential energy.
The E-BFMI, 0.01, is below the nominal threshold of 0.30 which suggests that HMC may have trouble exploring the target distribution.
If possible, try to reparameterize the model.

The following parameters had fewer than 0.001 effective draws per transition:
  sigma
Such low values indicate t

The diagnostics clearly indicate problems with estimating `sigma`. In this case, it is necessary to run the model with at least 150 warmup iterations to produce a good set of estimates when starting from the default initialization.

### Other approximation algorithms

We can follow the same pattern with Stan's ADVI algorithm by first using the `CmdStanModel.variationl` method. Because this algorithm is unstable and may fail to converge, we run it with argument `require_converged` set to `False`.  We also specify a seed, to avoid instabilities as well as for reproducibility.

In [10]:
vb_fit = model.variational(data=data_file, require_converged=False, seed=123)

21:47:12 - cmdstanpy - INFO - Chain [1] start processing
21:47:12 - cmdstanpy - INFO - Chain [1] done processing
21:47:12 - cmdstanpy - WARNING - The algorithm may not have converged.
Proceeding because require_converged is set to False


The ADVI algorithm provides estimates of all model parameters.

The `variational` method returns a `CmdStanVB` object, which similarly can construct a set of inits with the `.create_inits()` method:

In [11]:
vb_inits = vb_fit.create_inits()
for chain_init in vb_inits:
    print(chain_init)

{'beta': array([0.997148, 0.992516, 0.991829, 0.991095, 1.01057 ]), 'sigma': array(1.84433)}
{'beta': array([0.996249, 0.990954, 0.992313, 0.993369, 1.01155 ]), 'sigma': array(1.92087)}
{'beta': array([0.997361, 0.992357, 0.989631, 0.995749, 1.0083  ]), 'sigma': array(1.49741)}
{'beta': array([0.995738, 0.994643, 0.993908, 0.993482, 1.00921 ]), 'sigma': array(1.60191)}


Which can be passed to the `inits` keyword argument of the sample method.

In [12]:
mcmc_vb_inits_fit = model.sample(
    data=data_file, inits=vb_inits, iter_warmup=75, seed=12345
)

21:47:13 - cmdstanpy - INFO - CmdStan start processing


chain 1:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 2:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 3:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 4:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

21:47:13 - cmdstanpy - INFO - CmdStan done processing.
21:47:13 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'blr.stan', line 16, column 2 to column 45)
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'blr.stan', line 16, column 2 to column 45)
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'blr.stan', line 16, column 2 to column 45)
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'blr.stan', line 16, column 2 to column 45)
Consider re-running with show_console=True if the above output is unclear!


In [13]:
print(mcmc_vb_inits_fit.diagnose())

Processing csv files: /tmp/tmplo9k1zwz/blrwyskmpgi/blr-20250507214713_1.csv, /tmp/tmplo9k1zwz/blrwyskmpgi/blr-20250507214713_2.csv, /tmp/tmplo9k1zwz/blrwyskmpgi/blr-20250507214713_3.csv, /tmp/tmplo9k1zwz/blrwyskmpgi/blr-20250507214713_4.csv

Checking sampler transitions treedepth.
Treedepth satisfactory for all transitions.

Checking sampler transitions for divergences.
No divergent transitions found.

Checking E-BFMI - sampler transitions HMC potential energy.
E-BFMI satisfactory.

Effective sample size satisfactory.

Split R-hat values satisfactory all parameters.

Processing complete, no problems detected.



In [14]:
mcmc_vb_inits_fit.summary()

,Mean,MCSE,StdDev,5%,50%,95%,N_Eff,N_Eff/s,R_hat
lp__,-156.926000,0.062253,1.775690,-160.352000,-156.589000,-154.65400,813.61500,4596.70000,1.002930
beta[1],0.999489,0.000014,0.000955,0.997917,0.999494,1.00106,4623.05000,26118.90000,0.999487
beta[2],1.000240,0.000018,0.001148,0.998363,1.000260,1.00214,4167.40000,23544.60000,1.000340
beta[3],1.000420,0.000014,0.000941,0.998877,1.000420,1.00196,4680.46000,26443.30000,0.999984
beta[4],1.001150,0.000017,0.001082,0.999394,1.001130,1.00298,4017.75000,22699.20000,0.999891
beta[5],1.001570,0.000016,0.001072,0.999836,1.001590,1.00332,4777.06000,26989.10000,1.000730
sigma,0.962740,0.004720,0.072470,0.849300,0.959950,1.08129,235.93876,1332.98734,1.012130


The sampler estimates match the reference posterior with no diagnostic issues.

Inits can also be constructed from the `laplace` method:

In [15]:
laplace_inits = model.laplace_sample(data=data_file, seed=123).create_inits()

21:47:14 - cmdstanpy - INFO - Chain [1] start processing
21:47:14 - cmdstanpy - INFO - Chain [1] done processing
21:47:14 - cmdstanpy - INFO - Chain [1] start processing
21:47:14 - cmdstanpy - INFO - Chain [1] done processing


In [16]:
mcmc_laplace_inits_fit = model.sample(
    data=data_file, inits=laplace_inits, iter_warmup=75, seed=12345
)

21:47:14 - cmdstanpy - INFO - CmdStan start processing


chain 1:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 2:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 3:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 4:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

21:47:14 - cmdstanpy - INFO - CmdStan done processing.
21:47:14 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'blr.stan', line 16, column 2 to column 45)
Consider re-running with show_console=True if the above output is unclear!


In [17]:
print(mcmc_laplace_inits_fit.diagnose())

Processing csv files: /tmp/tmplo9k1zwz/blrw34ul8fk/blr-20250507214714_1.csv, /tmp/tmplo9k1zwz/blrw34ul8fk/blr-20250507214714_2.csv, /tmp/tmplo9k1zwz/blrw34ul8fk/blr-20250507214714_3.csv, /tmp/tmplo9k1zwz/blrw34ul8fk/blr-20250507214714_4.csv

Checking sampler transitions treedepth.
Treedepth satisfactory for all transitions.

Checking sampler transitions for divergences.
No divergent transitions found.

Checking E-BFMI - sampler transitions HMC potential energy.
E-BFMI satisfactory.

Effective sample size satisfactory.

Split R-hat values satisfactory all parameters.

Processing complete, no problems detected.



And the `optimize` method. Since optimizations attempts to return a posterior mode, this will initialize all chains at the same point, which is not typically ideal.

In [18]:
optimized_inits = model.optimize(data=data_file, seed=123).create_inits()

21:47:15 - cmdstanpy - INFO - Chain [1] start processing
21:47:15 - cmdstanpy - INFO - Chain [1] done processing


In [19]:
mcmc_optimize_inits_fit = model.sample(
    data=data_file, inits=optimized_inits, iter_warmup=75, seed=12345
)

21:47:15 - cmdstanpy - INFO - CmdStan start processing


chain 1:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 2:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 3:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 4:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

21:47:15 - cmdstanpy - INFO - CmdStan done processing.


In [20]:
print(mcmc_optimize_inits_fit.diagnose())

Processing csv files: /tmp/tmplo9k1zwz/blrj2jo18al/blr-20250507214715_1.csv, /tmp/tmplo9k1zwz/blrj2jo18al/blr-20250507214715_2.csv, /tmp/tmplo9k1zwz/blrj2jo18al/blr-20250507214715_3.csv, /tmp/tmplo9k1zwz/blrj2jo18al/blr-20250507214715_4.csv

Checking sampler transitions treedepth.
Treedepth satisfactory for all transitions.

Checking sampler transitions for divergences.
No divergent transitions found.

Checking E-BFMI - sampler transitions HMC potential energy.
E-BFMI satisfactory.

Effective sample size satisfactory.

Split R-hat values satisfactory all parameters.

Processing complete, no problems detected.



It is also possible to use the output of the `sample()` method to construct inits to be fed into a future sampling run:

In [21]:
first_mcmc_inits = model.sample(
    data=data_file, iter_warmup=75, seed=12345
).create_inits()

21:47:16 - cmdstanpy - INFO - CmdStan start processing


chain 1:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 2:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 3:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 4:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

21:47:16 - cmdstanpy - INFO - CmdStan done processing.
21:47:16 - cmdstanpy - WARNING - Some chains may have failed to converge.
	Chain 1 had 177 divergent transitions (17.7%)
	Chain 3 had 173 divergent transitions (17.3%)
	Chain 4 had 247 divergent transitions (24.7%)
	Use the "diagnose()" method on the CmdStanMCMC object to see further information.


In [22]:
second_mcmc_fit = model.sample(data=data_file, inits=first_mcmc_inits, iter_warmup=75, seed=12345)

21:47:16 - cmdstanpy - INFO - CmdStan start processing


chain 1:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 2:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 3:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 4:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

21:47:16 - cmdstanpy - INFO - CmdStan done processing.
21:47:16 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'blr.stan', line 16, column 2 to column 45)
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'blr.stan', line 16, column 2 to column 45)
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'blr.stan', line 16, column 2 to column 45)
Consider re-running with show_console=True if the above output is unclear!


In [23]:
print(second_mcmc_fit.diagnose())

Processing csv files: /tmp/tmplo9k1zwz/blrztx_euub/blr-20250507214716_1.csv, /tmp/tmplo9k1zwz/blrztx_euub/blr-20250507214716_2.csv, /tmp/tmplo9k1zwz/blrztx_euub/blr-20250507214716_3.csv, /tmp/tmplo9k1zwz/blrztx_euub/blr-20250507214716_4.csv

Checking sampler transitions treedepth.
Treedepth satisfactory for all transitions.

Checking sampler transitions for divergences.
No divergent transitions found.

Checking E-BFMI - sampler transitions HMC potential energy.
E-BFMI satisfactory.

Effective sample size satisfactory.

Split R-hat values satisfactory all parameters.

Processing complete, no problems detected.



We see that despite the initial sampling issues in the first MCMC run, the inits sourced from that run result in reasonable sampling in the second run.